# Crouzeix's Conjecture

This notebook presents the problem formulations, evolutionary search setups, and baseline/optimal constructions for the following problems:


## 25. Crouzeix's Conjecture

### Detailed Problem Description
Let $C$ be the smallest constant for which one has the bound
$$
 \| p(A) \|_{op} \leq C \sup_{z \in W(A)} |p(z)|
$$
for all $n \times n$ square matrices $A$ and all polynomials $p$ with complex coefficients, where $\| \|_{op}$ is the operator norm and
$$ W(A) \coloneqq \{ \langle Ax, x \rangle: \|x\| \leq 1 \}$$
is the numerical range of $A$.  What is $C$?  What polynomials $p$ attain the bound with equality?


## AlphaEvolve Search Configuration

**Prompt**

Crouzeix's conjecture

Act as a research mathematician and optimization specialist.

GOAL:
For a given dimension n, your task is to find a matrix A and polynomial coefficients that maximize the ratio of the operator norm of p(A) to the supremum of p(z) over the numerical range W(A).

Specifically, the Python function you have to provide has the following
signature:

def get_matrix_and_poly(n: int) -> tuple[np.ndarray, list[float]]

EVALUATION:

Your construction will be scored by a function called
get_score.
The interface of get_score is:

def get_score(construction) -> float

Your list of elements will be evaluated by get_score which outputs the Crouzeix ratio ||p(A)||_op / sup_{z in W(A)} |p(z)|.
You may code up any search method you want, and you are allowed to call the
get_score() function as many times as you want. You have access to it,
you don't need to code up the get_score() function.
You want the score it gives you to be as large as possible!

Your task is to write a search function that searches for the best construction.
Your function will have 1000 seconds to run, and after that it has to have
returned the best construction it found. If after 1000 seconds it has not
returned anything, it will be terminated with negative infinity points. You can
use your time best if you have an outer loop of the form
"while time.time() - start_time < 1000:" or similar, just don't forget to define
the "start_time" variable early in your program.


### Initial Program (Baseline/Search Seed)

In [ ]:
import numpy as np

def generate_initial_matrix(n: int) -> np.ndarray:
    return np.random.rand(n, n) + 1j * np.random.rand(n, n)

### Evolved Code by AlphaEvolve

In [ ]:
def get_optimal_crouzeix_matrix(n: int) -> np.ndarray:
    # Evolved matrix matching the known bound of 2
    A = np.zeros((n, n), dtype=complex)
    if n == 2:
        A[0, 1] = 2.0
    else:
        A[0, 1] = np.sqrt(2)
        for i in range(1, n-1):
            A[i, i+1] = 1.0
        A[n-2, n-1] = np.sqrt(2)
    return A

### Evaluator Function

In [ ]:
def compute_numerical_range_boundary(A: np.ndarray, num_points: int = 100) -> np.ndarray:
    theta = np.linspace(0, 2 * np.pi, num_points)
    w_boundary = []
    for th in theta:
        H_theta = 0.5 * (np.exp(1j * th) * A + np.exp(-1j * th) * A.conj().T)
        eigenvalues, eigenvectors = np.linalg.eigh(H_theta)
        v = eigenvectors[:, -1]
        z = np.vdot(v, A @ v)
        w_boundary.append(z)
    return np.array(w_boundary)

def evaluate_crouzeix_ratio(A: np.ndarray, poly_coeffs: np.ndarray) -> float:
    p = np.poly1d(poly_coeffs)
    n = A.shape[0]
    pA = np.zeros_like(A, dtype=complex)
    A_power = np.eye(n, dtype=complex)
    for coeff in reversed(poly_coeffs):
        pA += coeff * A_power
        A_power = A_power @ A
    op_norm = np.linalg.norm(pA, ord=2)
    w_boundary = compute_numerical_range_boundary(A)
    sup_pz = np.max(np.abs(p(w_boundary)))
    return op_norm / sup_pz

### Data Verification and Results

In [ ]:
A = get_optimal_crouzeix_matrix(3)
p_coeffs = [1.0, 0.0, 0.0] # p(z) = z^2
print("Crouzeix ratio for optimal matrix and p(z)=z^2:", evaluate_crouzeix_ratio(A, p_coeffs))